# Main

In [ ]:
import helpers.solver as solver
import helpers.postprocess as postprocess
import numpy as np

## Truss Validation

### Model Definition

The node numbering order and corresponding positions are as follows:

- **Odd-numbered nodes** are on the left truss plane: $x=0$
- **Even-numbered nodes** are on the right truss plane: $x=3$
- Along the roof truss length, the nodes are numbered in ascending order of **$y$**
- The roof height is represented by the **$z$ coordinate**: bottom chord nodes have $z=0$, while upper chord or ridge nodes have $z=2.598$ or $z=5.196$

| Node | Position $(X, Y, Z)$ m | Description |
|---|---|---|
| 1 | $(0,\ 0,\ 0)$ | Left end bottom node |
| 2 | $(3,\ 0,\ 0)$ | Right end bottom node |
| 3 | $(0,\ 4,\ 2.598)$ | Left front upper chord node |
| 4 | $(3,\ 4.5,\ 2.598)$ | Right front upper chord node |
| 5 | $(0,\ 6,\ 0)$ | Left bottom chord node |
| 6 | $(3,\ 6,\ 0)$ | Right bottom chord node |
| 7 | $(0,\ 9,\ 5.196)$ | Left node near the ridge |
| 8 | $(3,\ 9,\ 5.196)$ | Right node near the ridge |
| 9 | $(0,\ 12,\ 0)$ | Left bottom chord node |
| 10 | $(3,\ 12,\ 0)$ | Right bottom chord node |
| 11 | $(0,\ 13.5,\ 2.598)$ | Left rear upper chord node |
| 12 | $(3,\ 13.5,\ 2.598)$ | Right rear upper chord node |
| 13 | $(0,\ 18,\ 0)$ | Left end bottom node |
| 14 | $(3,\ 18,\ 0)$ | Right end bottom node |

It can also be understood in pairs of transverse nodes:

- Pair 1: 1–2, $y=0$
- Pair 2: 3–4, $y=4.5$
- Pair 3: 5–6, $y=6$
- Pair 4: 7–8, $y=9$
- Pair 5: 9–10, $y=12$
- Pair 6: 11–12, $y=13.5$
- Pair 7: 13–14, $y=18$

The element definitions are listed below.

All members are defined as **TRUSS** elements with **Material 1**, **Section 1**, and **MT = 0**.

| Element | Connectivity $(i, j)$ | Description |
|---|---:|---|
| 1 | (1, 5) | Left bottom chord member |
| 2 | (1, 3) | Left end diagonal member |
| 3 | (3, 5) | Left lower diagonal member |
| 4 | (3, 7) | Left upper chord member |
| 5 | (5, 7) | Left web diagonal member |
| 6 | (5, 9) | Left bottom chord member |
| 7 | (7, 9) | Left web diagonal member |
| 8 | (7, 11) | Left upper chord member |
| 9 | (9, 11) | Left lower diagonal member |
| 10 | (9, 13) | Left bottom chord member |
| 11 | (11, 13) | Left end diagonal member |
| 12 | (2, 6) | Right bottom chord member |
| 13 | (2, 4) | Right end diagonal member |
| 14 | (4, 6) | Right lower diagonal member |
| 15 | (4, 8) | Right upper chord member |
| 16 | (6, 8) | Right web diagonal member |
| 17 | (6, 10) | Right bottom chord member |
| 18 | (8, 10) | Right web diagonal member |
| 19 | (8, 12) | Right upper chord member |
| 20 | (10, 12) | Right lower diagonal member |
| 21 | (10, 14) | Right bottom chord member |
| 22 | (12, 14) | Right end diagonal member |
| 23 | (1, 2) | Transverse end bottom tie |
| 24 | (3, 4) | Transverse front upper tie |
| 25 | (5, 6) | Transverse bottom tie |
| 26 | (7, 8) | Transverse ridge tie |
| 27 | (9, 10) | Transverse bottom tie |
| 28 | (11, 12) | Transverse rear upper tie |
| 29 | (13, 14) | Transverse end bottom tie |
| 30 | (1, 4) | Front end transverse diagonal |
| 31 | (2, 3) | Front end transverse diagonal |
| 32 | (1, 6) | Lower transverse diagonal |
| 33 | (2, 5) | Lower transverse diagonal |
| 34 | (3, 8) | Front upper transverse diagonal |
| 35 | (4, 7) | Front upper transverse diagonal |
| 36 | (5, 10) | Middle lower transverse diagonal |
| 37 | (6, 9) | Middle lower transverse diagonal |
| 38 | (7, 12) | Upper transverse diagonal |
| 39 | (8, 11) | Upper transverse diagonal |
| 40 | (9, 14) | Rear lower transverse diagonal |
| 41 | (10, 13) | Rear lower transverse diagonal |
| 42 | (11, 14) | Rear end transverse diagonal |
| 43 | (12, 13) | Rear end transverse diagonal |

It can also be understood by member groups:

- **Left truss plane members:** 1--11  
- **Right truss plane members:** 12--22  
- **Transverse members:** 23--29  
- **Transverse bracing members:** 30--43

In addition, the cross-section info are listed below:

- Young's Modulus: $E=70 GPa$
- Area: $A=4000 mm^2$

In summary:

**The nodes are numbered from front to back in the $y$ direction, and for each transverse section, the left node is numbered first and the right node second. Thus, left-side nodes are odd and right-side nodes are even.**

---

### Validation File Description

In addition to the geometric modeling information above, four separate validation files were prepared to verify the implementation of different load cases. All four files use the same structural geometry, element connectivity, material, section, and boundary-condition definitions. The difference between them lies in the applied loading or imposed action.

#### 1. `validation_truss_nodal_loads.json`

This file is used to validate the nodal load function.

- A nodal load of $F_z = -100$ $kN$ is applied at node 11
- A nodal load of $F_x = 100$ $kN$ is applied at node 12
- All other nodal loads are zero
- No temperature load, fabrication error, or support displacement is applied in this case

This case is intended to verify whether the program can correctly assemble concentrated nodal forces into the global load vector and produce the corresponding structural response.

#### 2. `validation_truss_supp_disp.json`

This file is used to validate the support displacement function.

- An imposed support displacement of $u_z = 0.01$ $m$ is specified
- No external nodal load is applied
- No temperature load or fabrication error is applied in this case

This case is intended to verify whether the program can correctly convert prescribed support movement into the equivalent structural response and reaction output.

#### 3. `validation_truss_temperature.json`

This file is used to validate the temperature load function.

- A temperature change of $\Delta T = 30^\circ\mathrm{C}$ is assigned to all 43 truss elements
- No external nodal load is applied
- No support displacement or fabrication error is applied in this case

This case is intended to verify whether the program can correctly calculate thermal strain effects and assemble the corresponding equivalent load contribution.

#### 4. `validation_truss_fab_error.json`

This file is used to validate the fabrication error function.

- A fabrication error value of $\mathrm{ERROR} = 0.01$ $m$ on element 6 is assigned
- A fabrication error value of $\mathrm{ERROR} = -0.01$ $m$ on element 17 (Regarding the xz-plane symmetry of element 6) is assigned
- No external nodal load is applied
- No support displacement or temperature load is applied in this case

This case is intended to verify whether the program can correctly account for initial length error effects in truss members and include them in the analysis.

#### Summary

These four validation files were created so that each loading function can be checked independently before applying the program to more complicated structural cases. In other words:

- `validation_truss_nodal_loads.json` verifies nodal force input
- `validation_truss_supp_disp.json` verifies imposed support displacement
- `validation_truss_temperature.json` verifies temperature loading
- `validation_truss_fab_error.json` verifies fabrication error loading

By separating the validation cases in this way, it becomes easier to confirm that each loading module is implemented correctly and that the corresponding response is physically reasonable.

---

### Validation Nodal Loads

In [ ]:
filepath = 'inputs/validation_truss_nodal_loads.json'

result = solver.truss_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")


### Validation Support Displacement

In [ ]:
filepath = 'inputs/validation_truss_supp_disp.json'

result = solver.truss_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")

### Validation Temperature Loads

In [ ]:
filepath = 'inputs/validation_truss_temperature.json'

result = solver.truss_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")

### Validation Fabrication Error Loads

In [ ]:
filepath = 'inputs/validation_truss_fab_error.json'

result = solver.truss_solver(filepath)
pp = postprocess.output(filepath, result, show_member_ids=True, table_output="xlsx")

---

## Frame Validation

### Model Definition

The frame validation model is a **three-dimensional four-column frame structure** with a square plan and multiple floor levels.

The node numbering order and corresponding positions are as follows:

- The structure has a square plan of **$8 \times 8$ m**
- The four corner columns are located at:
  - $(0,\ 0)$
  - $(0,\ 8)$
  - $(8,\ 8)$
  - $(8,\ 0)$
- The structure has five elevation levels in the **$z$ direction**:
  - $z=0$
  - $z=6$
  - $z=12$
  - $z=18$
  - $z=24$
- At each elevation, the nodes are numbered in a consistent order around the frame:
  - first $(0,\ 0)$
  - second $(0,\ 8)$
  - third $(8,\ 8)$
  - fourth $(8,\ 0)$
- Therefore, nodes are numbered **floor by floor from bottom to top**

| Node | Position $(X, Y, Z)$ m | Description |
|---|---|---|
| 1 | $(0,\ 0,\ 0)$ | Bottom node at corner 1 |
| 2 | $(0,\ 8,\ 0)$ | Bottom node at corner 2 |
| 3 | $(8,\ 8,\ 0)$ | Bottom node at corner 3 |
| 4 | $(8,\ 0,\ 0)$ | Bottom node at corner 4 |
| 5 | $(0,\ 0,\ 6)$ | First-floor node at corner 1 |
| 6 | $(0,\ 8,\ 6)$ | First-floor node at corner 2 |
| 7 | $(8,\ 8,\ 6)$ | First-floor node at corner 3 |
| 8 | $(8,\ 0,\ 6)$ | First-floor node at corner 4 |
| 9 | $(0,\ 0,\ 12)$ | Second-floor node at corner 1 |
| 10 | $(0,\ 8,\ 12)$ | Second-floor node at corner 2 |
| 11 | $(8,\ 8,\ 12)$ | Second-floor node at corner 3 |
| 12 | $(8,\ 0,\ 12)$ | Second-floor node at corner 4 |
| 13 | $(0,\ 0,\ 18)$ | Third-floor node at corner 1 |
| 14 | $(0,\ 8,\ 18)$ | Third-floor node at corner 2 |
| 15 | $(8,\ 8,\ 18)$ | Third-floor node at corner 3 |
| 16 | $(8,\ 0,\ 18)$ | Third-floor node at corner 4 |
| 17 | $(0,\ 0,\ 24)$ | Roof-level node at corner 1 |
| 18 | $(0,\ 8,\ 24)$ | Roof-level node at corner 2 |
| 19 | $(8,\ 8,\ 24)$ | Roof-level node at corner 3 |
| 20 | $(8,\ 0,\ 24)$ | Roof-level node at corner 4 |

It can also be understood by floor levels:

- Ground level: 1--4, $z=0$
- First floor: 5--8, $z=6$
- Second floor: 9--12, $z=12$
- Third floor: 13--16, $z=18$
- Roof level: 17--20, $z=24$

The element definitions are listed below.

All members are defined as **FRAME** elements with **Material 1**, **Section 1**, and **MT = 0**.

| Element | Connectivity $(i, j)$ | Description |
|---|---:|---|
| 1 | (1, 5) | Column member at corner 1, level 0--6 |
| 2 | (2, 6) | Column member at corner 2, level 0--6 |
| 3 | (3, 7) | Column member at corner 3, level 0--6 |
| 4 | (4, 8) | Column member at corner 4, level 0--6 |
| 5 | (5, 9) | Column member at corner 1, level 6--12 |
| 6 | (6, 10) | Column member at corner 2, level 6--12 |
| 7 | (7, 11) | Column member at corner 3, level 6--12 |
| 8 | (8, 12) | Column member at corner 4, level 6--12 |
| 9 | (9, 13) | Column member at corner 1, level 12--18 |
| 10 | (10, 14) | Column member at corner 2, level 12--18 |
| 11 | (11, 15) | Column member at corner 3, level 12--18 |
| 12 | (12, 16) | Column member at corner 4, level 12--18 |
| 13 | (13, 17) | Column member at corner 1, level 18--24 |
| 14 | (14, 18) | Column member at corner 2, level 18--24 |
| 15 | (15, 19) | Column member at corner 3, level 18--24 |
| 16 | (16, 20) | Column member at corner 4, level 18--24 |
| 17 | (5, 6) | First-floor beam |
| 18 | (6, 7) | First-floor beam |
| 19 | (7, 8) | First-floor beam |
| 20 | (8, 5) | First-floor beam |
| 21 | (9, 10) | Second-floor beam |
| 22 | (10, 11) | Second-floor beam |
| 23 | (11, 12) | Second-floor beam |
| 24 | (12, 9) | Second-floor beam |
| 25 | (13, 14) | Third-floor beam |
| 26 | (14, 15) | Third-floor beam |
| 27 | (15, 16) | Third-floor beam |
| 28 | (16, 13) | Third-floor beam |
| 29 | (17, 18) | Roof beam |
| 30 | (18, 19) | Roof beam |
| 31 | (19, 20) | Roof beam |
| 32 | (20, 17) | Roof beam |

It can also be understood by member groups:

- **Column members:** 1--16  
- **First-floor beam members:** 17--20  
- **Second-floor beam members:** 21--24  
- **Third-floor beam members:** 25--28  
- **Roof beam members:** 29--32  

The support conditions are defined as follows:

- Nodes **1, 2, 3, and 4** are the four ground-contact nodes
- These four nodes are all defined as **fixed supports**
- Therefore, all six degrees of freedom at these nodes are restrained:
  - $U_x=0$
  - $U_y=0$
  - $U_z=0$
  - $R_x=0$
  - $R_y=0$
  - $R_z=0$

In summary:

**The nodes are numbered floor by floor from bottom to top. At each elevation, the four corner nodes are numbered in a consistent order around the square frame. The model consists of four vertical columns and four horizontal beams at each upper level, forming a regular three-dimensional frame validation structure.**

---

### Validation Nodal Loads

In [ ]:
filepath = 'inputs/validation_frame_nodal_loads.json'

result = solver.frame_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")

### Validation Member Loads

In [ ]:
filepath = 'inputs/validation_frame_member_loads.json' # Currently unavailable

result = solver.frame_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")

### Validation Moment Release

In [ ]:
filepath = 'inputs/validation_frame_moment_release.json'

#result = solver.frame_solver(filepath)
#pp = postprocess.output(filepath, result, table_output="xlsx")

### Validation Support Displacement

In [ ]:
filepath = 'inputs/validation_frame_supp_disp.json'

result = solver.frame_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")

### Validation Temperature Loads

In [ ]:
filepath = 'inputs/validation_frame_temperature.json'

result = solver.frame_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")


### Validation Fabrication Error Loads

In [ ]:
filepath = 'inputs/validation_frame_fab_error.json'

result = solver.frame_solver(filepath)
pp = postprocess.output(filepath, result, table_output="xlsx")
